# Text2Art: text-conditioned DCGAN
Prompt → CLIP text embedding → DCGAN generator → 64×64 artwork.

`pip install torch torchvision transformers pandas pyarrow pillow matplotlib imageio tqdm`

### Imports

In [ ]:
import os, time, random
from datetime import datetime

import torch
import torch.nn as nn
import torchvision.utils as vutils
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from utils import weights_init
from dcgan_model import Generator, Discriminator
from text_encoder import CLIPTextEncoder
from art_dataset import prepare_data, Text2ArtDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

### Config

In [ ]:
# a folder of .parquet shards (searched recursively), one file, a glob, or a list.
# Point at ONE split folder so test/validation shards don't get mixed into training.
DATA_SOURCE  = "./datasets/data/samples/train"
OUTPUT_DIR   = "./generated_images/"
MODEL_DIR    = "./saved_models/"
CACHE_DIR    = "./cache/"
for d in (OUTPUT_DIR, MODEL_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

# data
IMAGE_SIZE  = 64          # the DCGAN architecture is fixed to 64x64
CROP        = "center"    # "center": cut middle square | "squash": resize whole image into a square
MAX_SAMPLES = None        # total cap across all shards, e.g. 2000 for a quick smoke test
NUM_THREADS = 8           # image-decoding threads

# model
noise_dim     = 100
embed_out_dim = 128
# embed_dim is set automatically from the text encoder (512 for CLIP ViT-B/32)

# training
batch_size    = 128
num_epochs    = 200
learning_rate = 2e-4
real_label    = 0.9       # one-sided label smoothing for the discriminator
l1_coef       = 10        # pixel loss vs. the paired image; high values -> blurry "average" art
l2_coef       = 100       # feature-matching loss
sample_every  = 5         # epochs between sample grids
ckpt_every    = 10        # epochs between checkpoints

date = datetime.now().strftime("%Y%m%d")

### Data
Images are decoded + resized once and cached; prompts are embedded once with a frozen CLIP text encoder and cached.

In [ ]:
encoder = CLIPTextEncoder("openai/clip-vit-base-patch32", device=device)
embed_dim = encoder.embed_dim

images, embeds, prompts = prepare_data(
    DATA_SOURCE, encoder, image_size=IMAGE_SIZE, crop=CROP,
    cache_dir=CACHE_DIR, max_samples=MAX_SAMPLES, num_threads=NUM_THREADS)
print(f"{len(images)} images {tuple(images.shape[1:])}, embeddings {tuple(embeds.shape)}")

train_dataset = Text2ArtDataset(images, embeds, flip=True)
# drop_last: BatchNorm1d in the text-embedding layer fails on a batch of size 1
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=0, drop_last=True, pin_memory=(device.type == "cuda"))
print("No of batches:", len(train_loader))

# sanity check: a few training images with their prompts
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, i in zip(axes, range(6)):
    ax.imshow(images[i].permute(1, 2, 0).numpy()); ax.axis("off")
    ax.set_title(prompts[i][:30], fontsize=7)
plt.show()

### Models

In [ ]:
generator = Generator(channels=3, noise_dim=noise_dim, embed_dim=embed_dim, embed_out_dim=embed_out_dim).to(device)
generator.apply(weights_init)
discriminator = Discriminator(channels=3, embed_dim=embed_dim, embed_out_dim=embed_out_dim).to(device)
discriminator.apply(weights_init)

optimizer_G = torch.optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

criterion = nn.BCELoss()
l2_loss, l1_loss = nn.MSELoss(), nn.L1Loss()
D_losses, G_losses = [], []

config = dict(noise_dim=noise_dim, embed_dim=embed_dim, embed_out_dim=embed_out_dim,
              image_size=IMAGE_SIZE, encoder=encoder.model_name)

def save_checkpoint(path):
    torch.save({"generator": generator.state_dict(),
                "discriminator": discriminator.state_dict(),
                "config": config}, path)

# fixed prompts + noise so progress across epochs is directly comparable
n_viz = 8
fixed_real   = (images[:n_viz].float() / 127.5 - 1).to(device)
fixed_embeds = embeds[:n_viz].to(device)
fixed_noise  = torch.randn(n_viz, noise_dim, 1, 1, device=device)

@torch.no_grad()
def save_samples(epoch):
    generator.eval()
    fake = generator(fixed_noise, fixed_embeds)
    generator.train()
    # top row: real art for those prompts, bottom row: generated
    vutils.save_image(torch.cat([fixed_real, fake]),
                      os.path.join(OUTPUT_DIR, f"output_{date}_epoch_{epoch:04d}.png"),
                      nrow=n_viz, normalize=True, value_range=(-1, 1))

### Training
Matching-aware conditional GAN (GAN-CLS): the discriminator sees *(real image, its text)* as real, and both *(real image, wrong text)* and *(fake image, text)* as fake.

In [ ]:
start_time = time.time()

for epoch in range(1, num_epochs + 1):
    t0 = time.time()
    d_sum = g_sum = 0.0

    for batch in train_loader:
        images_b = batch["image"].to(device, non_blocking=True)
        embeds_b = batch["embed"].to(device, non_blocking=True)
        B = images_b.size(0)

        # images paired with someone else's text (shift by a random non-zero offset)
        wrong_images = torch.roll(images_b, shifts=random.randint(1, B - 1), dims=0)

        # ---------------- Discriminator ----------------
        optimizer_D.zero_grad()
        noise = torch.randn(B, noise_dim, 1, 1, device=device)
        fake_images = generator(noise, embeds_b)

        real_out,  real_act = discriminator(images_b, embeds_b)
        wrong_out, _        = discriminator(wrong_images, embeds_b)
        fake_out,  _        = discriminator(fake_images.detach(), embeds_b)

        d_loss = (criterion(real_out,  torch.full_like(real_out, real_label))
                  + 0.5 * (criterion(wrong_out, torch.zeros_like(wrong_out))
                           + criterion(fake_out,  torch.zeros_like(fake_out))))
        d_loss.backward()
        optimizer_D.step()

        # ---------------- Generator ----------------
        optimizer_G.zero_grad()
        out_fake, act_fake = discriminator(fake_images, embeds_b)

        g_bce = criterion(out_fake, torch.ones_like(out_fake))
        g_l1  = l1_coef * l1_loss(fake_images, images_b)
        g_l2  = l2_coef * l2_loss(act_fake.mean(0), real_act.detach().mean(0))
        g_loss = g_bce + g_l1 + g_l2
        g_loss.backward()
        optimizer_G.step()

        D_losses.append(d_loss.item()); G_losses.append(g_loss.item())
        d_sum += d_loss.item(); g_sum += g_loss.item()

    n = len(train_loader)
    print(f"Epoch {epoch:3d}/{num_epochs}  loss_D {d_sum/n:.4f}  loss_G {g_sum/n:.4f}  ({time.time()-t0:.1f}s)")

    if epoch == 1 or epoch % sample_every == 0:
        save_samples(epoch)
    if epoch % ckpt_every == 0:
        save_checkpoint(os.path.join(MODEL_DIR, "text2art_latest.pt"))

save_checkpoint(os.path.join(MODEL_DIR, f"text2art_{date}.pt"))
save_checkpoint(os.path.join(MODEL_DIR, "text2art_latest.pt"))
print(f"Total train time: {time.time() - start_time:.1f}s")

### Loss curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(D_losses, label="D", alpha=0.7)
plt.plot(G_losses, label="G", alpha=0.7)
plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend(); plt.title("Training losses")
plt.savefig(os.path.join(OUTPUT_DIR, f"losses_{date}.png"))   # save BEFORE show, otherwise the file is blank
plt.show()

### Generate art from your own prompt
Each row is one prompt, each column a different noise vector.

In [ ]:
from generate import generate
from IPython.display import Image, display

my_prompts = [
    "expressionist painting of a lonely mountain at dusk",
    "pencil sketch of birds over water",
]
generate(my_prompts, os.path.join(MODEL_DIR, "text2art_latest.pt"),
         n=8, out_path="my_art.png", scale=4, encoder=encoder)
display(Image("my_art.png"))

### (Optional) Training progress GIF

In [ ]:
import imageio
files = sorted(f for f in os.listdir(OUTPUT_DIR) if f.startswith(f"output_{date}_epoch_"))
frames = [imageio.imread(os.path.join(OUTPUT_DIR, f)) for f in files]
gif_path = os.path.join(OUTPUT_DIR, f"progress_{date}.gif")
imageio.mimsave(gif_path, frames, fps=2)
display(Image(gif_path))